# FeatureImpact — Notebook 1: Synthetic A/B Experiment Data Generation

**Objective:** Generate realistic synthetic A/B experiment data simulating a SaaS product feature rollout.

We simulate a scenario where Atlassian rolls out a new Jira dashboard feature to a treatment group and measures:
- **Conversion rate** (did the user complete a key action?)
- **Session duration** (time spent in the product, in seconds)
- **Click-through rate** (CTR on in-product prompts)

The treatment group has a small but realistic lift to reflect a genuine product improvement.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

np.random.seed(42)  # Reproducibility

print("Libraries loaded successfully.")

## 2. Experiment Configuration

Define all parameters in one place — easy to tweak for different scenarios.

In [ ]:
# ─── Experiment Parameters ───────────────────────────────────────────────────
EXPERIMENT_NAME     = "jira_dashboard_v2_rollout"
N_CONTROL           = 2000       # Users in control group
N_TREATMENT         = 2000       # Users in treatment group
EXPERIMENT_DAYS     = 14         # Duration of experiment

# Conversion rate (baseline and lift)
CONTROL_CONV_RATE   = 0.12       # 12% baseline conversion
TREATMENT_LIFT      = 0.03       # +3% absolute lift in treatment

# Session duration (seconds)
CONTROL_SESSION_MEAN   = 320     # ~5.3 min average session
CONTROL_SESSION_STD    = 95
TREATMENT_SESSION_MEAN = 345     # ~5.75 min (treatment has longer sessions)
TREATMENT_SESSION_STD  = 90

# Click-through rate
CONTROL_CTR         = 0.08
TREATMENT_CTR       = 0.10

# Missing value / outlier rates (realistic noise)
MISSING_RATE        = 0.02       # 2% of rows will have missing session data
OUTLIER_RATE        = 0.01       # 1% of rows will have extreme session values

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Total users: {N_CONTROL + N_TREATMENT:,}")
print(f"Expected control conversion: {CONTROL_CONV_RATE:.0%}")
print(f"Expected treatment conversion: {CONTROL_CONV_RATE + TREATMENT_LIFT:.0%}")

## 3. Generate User-Level Data

In [ ]:
def generate_group(group_label, n_users, conv_rate, session_mean,
                   session_std, ctr, start_date="2024-11-25"):
    """
    Generate synthetic user-level experiment data for one group.
    
    Returns a DataFrame with one row per user.
    """
    start = pd.Timestamp(start_date)

    df = pd.DataFrame({
        # Unique user identifier
        "user_id": [f"{group_label[:1].upper()}{str(i).zfill(5)}" for i in range(n_users)],

        # Assignment
        "group": group_label,

        # Random entry day within experiment window
        "entry_date": [start + pd.Timedelta(days=int(d))
                       for d in np.random.randint(0, EXPERIMENT_DAYS, n_users)],

        # Primary metric: conversion (binary)
        "converted": np.random.binomial(1, conv_rate, n_users),

        # Secondary metric: session duration (continuous, normally distributed)
        "session_duration_sec": np.random.normal(session_mean, session_std, n_users).clip(30),

        # Secondary metric: click-through (binary)
        "clicked_prompt": np.random.binomial(1, ctr, n_users),

        # User segment (for subgroup analysis)
        "plan_type": np.random.choice(
            ["free", "standard", "premium"],
            size=n_users,
            p=[0.50, 0.35, 0.15]
        ),

        # Device type
        "device": np.random.choice(
            ["desktop", "mobile", "tablet"],
            size=n_users,
            p=[0.65, 0.28, 0.07]
        ),
    })

    return df


# Generate both groups
control_df   = generate_group("control",   N_CONTROL,   CONTROL_CONV_RATE,
                               CONTROL_SESSION_MEAN,   CONTROL_SESSION_STD,   CONTROL_CTR)

treatment_df = generate_group("treatment", N_TREATMENT, CONTROL_CONV_RATE + TREATMENT_LIFT,
                               TREATMENT_SESSION_MEAN, TREATMENT_SESSION_STD, TREATMENT_CTR)

# Combine
df = pd.concat([control_df, treatment_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle rows

print(f"Dataset shape: {df.shape}")
df.head(10)

## 4. Inject Realistic Noise

Real data is never clean. We add missing values and outliers to make the dataset more authentic.

In [ ]:
# ─── Missing values in session duration ──────────────────────────────────────
missing_idx = df.sample(frac=MISSING_RATE, random_state=7).index
df.loc[missing_idx, "session_duration_sec"] = np.nan

# ─── Outliers: extreme session values (bots / power users) ───────────────────
outlier_idx = df.sample(frac=OUTLIER_RATE, random_state=13).index
df.loc[outlier_idx, "session_duration_sec"] = np.random.uniform(1800, 7200, len(outlier_idx))

print(f"Missing session values injected: {df['session_duration_sec'].isna().sum()}")
print(f"Outlier rows injected: {len(outlier_idx)}")
print(f"\nSession duration stats after noise:")
print(df['session_duration_sec'].describe().round(2))

## 5. Sanity Check — Group Balance

In [ ]:
print("=" * 55)
print("GROUP BALANCE CHECK")
print("=" * 55)

summary = df.groupby("group").agg(
    n_users       = ("user_id",            "count"),
    conv_rate     = ("converted",          "mean"),
    avg_session   = ("session_duration_sec","mean"),
    ctr           = ("clicked_prompt",     "mean"),
).round(4)

print(summary)
print("\nPlan type distribution:")
print(pd.crosstab(df['group'], df['plan_type'], normalize='index').round(3))

## 6. Quick Visual — Conversion Rate Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"Experiment: {EXPERIMENT_NAME} — Group Comparison", fontsize=14, fontweight="bold")

palette = {"control": "#4C9BE8", "treatment": "#F4845F"}

# Conversion rate bar
conv_rates = df.groupby("group")["converted"].mean().reset_index()
axes[0].bar(conv_rates["group"], conv_rates["converted"],
            color=[palette[g] for g in conv_rates["group"]], edgecolor="white", width=0.5)
axes[0].set_title("Conversion Rate", fontweight="bold")
axes[0].set_ylabel("Rate")
axes[0].set_ylim(0, 0.25)
for i, v in enumerate(conv_rates["converted"]):
    axes[0].text(i, v + 0.005, f"{v:.1%}", ha="center", fontweight="bold")

# Session duration box plot
clean = df.dropna(subset=["session_duration_sec"])
groups = [clean[clean["group"]=="control"]["session_duration_sec"],
          clean[clean["group"]=="treatment"]["session_duration_sec"]]
bp = axes[1].boxplot(groups, labels=["control", "treatment"], patch_artist=True,
                     medianprops=dict(color="white", linewidth=2))
for patch, color in zip(bp['boxes'], [palette["control"], palette["treatment"]]):
    patch.set_facecolor(color)
axes[1].set_title("Session Duration (sec)", fontweight="bold")
axes[1].set_ylabel("Seconds")

# CTR bar
ctr_rates = df.groupby("group")["clicked_prompt"].mean().reset_index()
axes[2].bar(ctr_rates["group"], ctr_rates["clicked_prompt"],
            color=[palette[g] for g in ctr_rates["group"]], edgecolor="white", width=0.5)
axes[2].set_title("Click-Through Rate", fontweight="bold")
axes[2].set_ylabel("Rate")
axes[2].set_ylim(0, 0.20)
for i, v in enumerate(ctr_rates["clicked_prompt"]):
    axes[2].text(i, v + 0.003, f"{v:.1%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("nb1_group_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

## 7. Save Dataset to CSV

In [ ]:
os.makedirs("data", exist_ok=True)
df.to_csv("data/ab_experiment_raw.csv", index=False)

print("Dataset saved to: data/ab_experiment_raw.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print("\nSample rows:")
df.sample(5, random_state=1)

## Summary

| Item | Detail |
|---|---|
| Rows generated | 4,000 (2,000 per group) |
| Primary metric | Conversion rate (binary) |
| Secondary metrics | Session duration, CTR |
| Noise added | 2% missing values, 1% outliers |
| Output file | `data/ab_experiment_raw.csv` |

**Next:** Load `data/ab_experiment_raw.csv` in Notebook 2 for Exploratory Data Analysis.